# ECG Dataset Exploratory Data Analysis (EDA)

This notebook provides an overview of the PTB‑XL ECG dataset used in the balancing experiments. It loads the full dataset, displays class distributions, and visualises basic signal statistics for normal and abnormal classes.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
project_root = os.path.abspath('..')
import sys
sys.path.append(project_root)

from data_management.dataset_factory import DatasetFactory

# Load full dataset (low‑resolution) for each balance mode
def load_data(balance_mode):
    _, val_ds, test_ds, loader = DatasetFactory.create_datasets(
        dataset_type='ptbxl', download=False, resolution='lr', balance_mode=balance_mode
    )
    return val_ds, test_ds, loader

# Example: load the 'average' balanced split (you can change mode)
val_ds, test_ds, loader = load_data('average')
class_names = loader.label_encoder.classes
print('Classes:', class_names)


In [ ]:
# Extract multi‑label vectors for a split
def extract_labels(ds):
    labels = []
    for rec_id in ds.record_ids:
        row = loader.metadata_df.loc[rec_id]
        diag_classes = loader.parser.get_diagnostic_classes(row.get('scp_codes', {}))
        encoded = loader.label_encoder.encode(diag_classes)
        labels.append(encoded)
    return np.array(labels)

val_labels = extract_labels(val_ds)
test_labels = extract_labels(test_ds)

# Per‑class support (how many records contain each label)
support_val = val_labels.sum(axis=0)
support_test = test_labels.sum(axis=0)
df_support = pd.DataFrame({
    'Class': class_names,
    'Val Support': support_val,
    'Test Support': support_test
})
display(df_support)


In [ ]:
# Visualise the support as bar plots
fig, ax = plt.subplots(1, 2, figsize=(12,5))
sns.barplot(x='Class', y='Val Support', data=df_support, ax=ax[0])
ax[0].set_title('Validation Set – Class Support')
sns.barplot(x='Class', y='Test Support', data=df_support, ax=ax[1])
ax[1].set_title('Test Set – Class Support')
plt.tight_layout()
plt.show()


## Signal Examples

Below we visualise a few example ECG recordings for the normal (NORM) class and an abnormal class (e.g., MI).

In [ ]:
def plot_ecg(signal, title='ECG'):
    plt.figure(figsize=(12,3))
    plt.plot(signal.T)  # each row is a lead
    plt.title(title)
    plt.xlabel('Sample')
    plt.ylabel('Amplitude')
    plt.show()

# Find an example for a given class
def get_example(idx, class_name):
    class_idx = list(class_names).index(class_name)
    rec_idx = np.where(val_labels[:, class_idx]==1)[0][idx]
    signal, _ = val_ds[rec_idx]
    return signal

norm_signal = get_example(0, 'NORM')
mi_signal   = get_example(0, 'MI')  # choose another abnormal class if you wish
plot_ecg(norm_signal, title='Normal (NORM) Example')
plot_ecg(mi_signal,   title='Abnormal (MI) Example')
